# Aula 1 — Detetive de Incidentes
### Agentes de IA: fundamentos e primeira análise com LLM

**Nosso caso:** um assistente que recebe relatos fictícios de falhas, identifica sintomas e organiza uma análise inicial. Nas próximas aulas, vamos conectá-lo ao LangGraph e a ferramentas de consulta.

**Como vamos trabalhar:** acompanhe a demonstração, execute cada célula e depois tente os desafios. Não é necessário memorizar as bibliotecas: concentre-se no que entra, no que o código faz e no que retorna.

> **Limite do projeto:** investigar e explicar. Nenhuma célula deve realizar deploy, rollback ou alterar sistemas reais.

## Objetivo e percurso de hoje

Ao final, você deverá conseguir:

1. **Distinguir** chatbot, workflow determinístico e agente com ferramentas.
2. **Reconhecer** LLM, instruções, mensagens/estado e ferramentas no código.
3. **Enviar** uma solicitação a um LLM, testar um **chat com Gradio** e orientar sua resposta.
4. **Definir um contrato de saída** com Pydantic e analisar diferentes relatos.

| Parte | Slides de referência | Resultado |
|---|---|---|
| Conceitos na prática | 4 e 7 | Chat com Gradio e comparação entre três abordagens |
| Primeiro LLM e instruções | 10–13 | Mensagens e regras explícitas |
| Saída estruturada | 14–15 | Objeto Python validado |
| Testes e reuso | 16–20 | Função `analisar_incidente(texto)` |

**Hoje não vamos construir o grafo do agente.** Essa é a prática da Aula 2.

## 0. Preparar o ambiente — OpenAI **ou** Google Gemini

Vamos usar **Python, LangChain, Pydantic e Gradio**. O Gradio será a interface visual para o chat; o LangChain chamará o modelo escolhido. **Você só precisa de uma das duas chaves**, não das duas.

1. Execute a célula de instalação, agora com **`langchain-openai` e `langchain-google-genai`**, além do Gradio.
2. Execute a célula de configuração e escolha **1 — OpenAI** ou **2 — Gemini**. Se pressionar Enter sem escolher, o padrão será **Gemini**.
3. Se a chave ainda não estiver no ambiente, informe-a no **campo oculto** solicitado pelo notebook. A chave não será exibida.
4. Execute os exercícios normalmente: as demais células usam o mesmo objeto `modelo`, independentemente da escolha.

| Provedor | Onde criar a chave | Variável reconhecida | Modelo inicial |
|---|---|---|---|
| **OpenAI** | [OpenAI API Keys](https://platform.openai.com/api-keys) | `OPENAI_API_KEY` | `gpt-4.1-mini` |
| **Google Gemini** | [Google AI Studio](https://aistudio.google.com/api-keys) | `GOOGLE_API_KEY` ou `GEMINI_API_KEY` | `gemini-3.5-flash-lite` |

**Opcional:** personalize os modelos com as variáveis `OPENAI_MODEL` ou `GEMINI_MODEL`, antes de executar a configuração. Para seleção automática, use `LLM_PROVIDER=openai` ou `LLM_PROVIDER=gemini`. Um arquivo `.env` local pode armazenar essas configurações, mas **não deve ser enviado para repositórios ou compartilhado**.

> **Atenção:** ambos os provedores têm limites e condições próprios de uso. O Gemini pode oferecer cota gratuita; ela não é ilimitada. O modelo e a função de tool calling/saída estruturada precisam estar disponíveis para a sua chave.
>
> O Gradio será iniciado localmente (`share=False`). Se você trocar o provedor durante a aula, feche o chat com `chat_gradio.close()`, reexecute a configuração e execute novamente as células que criam objetos derivados de `modelo` (como `modelo_com_ferramentas` e `classificador`).


In [ ]:
%pip install -qU langchain langchain-openai langchain-google-genai "pydantic>=2" python-dotenv "gradio>=6,<7"

In [ ]:
# Configuração comum: escolha UM provedor e informe apenas a chave correspondente.
# Credenciais são recebidas por getpass() (campo oculto) ou lidas de variáveis de ambiente.
import os
import json
from getpass import getpass
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool

load_dotenv(".env")

# Também é possível definir LLM_PROVIDER=openai ou LLM_PROVIDER=gemini no ambiente.
escolha = os.getenv("LLM_PROVIDER", "").strip().lower()
if not escolha:
    escolha = input("Escolha o provedor [1] OpenAI ou [2] Gemini (padrão 2): ").strip().lower()

if escolha in ("1", "openai"):
    PROVEDOR = "openai"
    CHAVE_AMBIENTE = "OPENAI_API_KEY"
    MODELO_ID = os.getenv("OPENAI_MODEL", "gpt-4.1-mini").strip()
    ROTULO_PROVEDOR = "OpenAI"
elif escolha in ("", "2", "gemini", "google", "google_genai"):
    PROVEDOR = "google_genai"   # API do Gemini no Google AI Studio (não Vertex AI).
    CHAVE_AMBIENTE = "GOOGLE_API_KEY"
    MODELO_ID = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite").strip()
    ROTULO_PROVEDOR = "Google Gemini"
else:
    raise ValueError("Provedor inválido. Escolha 1/OpenAI ou 2/Gemini.")

if PROVEDOR == "openai":
    if not os.getenv("OPENAI_API_KEY"):
        chave = getpass("Informe sua OPENAI_API_KEY (entrada oculta): ").strip()
        if not chave:
            raise ValueError("É necessária uma API key da OpenAI para continuar.")
        os.environ["OPENAI_API_KEY"] = chave
else:
    # A integração Google aceita GOOGLE_API_KEY ou GEMINI_API_KEY.
    # Se ambas estiverem definidas, GOOGLE_API_KEY tem prioridade.
    if not (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")):
        chave = getpass("Informe sua API key do Gemini (entrada oculta): ").strip()
        if not chave:
            raise ValueError("É necessária uma API key do Google AI Studio para continuar.")
        os.environ["GOOGLE_API_KEY"] = chave

if not MODELO_ID:
    raise ValueError("O identificador do modelo não pode ser vazio.")

# A lógica das demais células é igual para ambos os provedores.
CHAT_MODEL = f"{PROVEDOR}:{MODELO_ID}"
modelo = init_chat_model(CHAT_MODEL, temperature=0)
print("Provedor escolhido:", ROTULO_PROVEDOR)
print("Modelo configurado:", CHAT_MODEL)
print("Chave de API disponível: sim (valor não exibido)")
# A primeira requisição REAL ao provedor ocorre em modelo.invoke(), adiante.


---
## 1. Chatbot, workflow e agente (slide 4)

Os três podem receber a mesma solicitação, mas **o próximo passo da execução é definido de maneiras diferentes**.

| Abordagem | Quem define o próximo passo? | Neste notebook |
|---|---|---|
| **Chatbot** | O modelo responde à mensagem | Chamada de LLM e chat visual com Gradio |
| **Workflow determinístico** | Regras escritas no programa | Uma função com `if` |
| **Agente** | O modelo pode escolher uma ferramenta autorizada | Uma consulta simulada, sem LangGraph |

Use o mesmo relato fictício e compare. A finalidade aqui é **enxergar a diferença**, não construir uma solução completa de cada tipo.

**OpenAI e Gemini — exibição do texto:** a mensagem de resposta pode guardar `content` como string (frequente na OpenAI) ou como lista de blocos de conteúdo e metadados (possível no Gemini). Use **`.text`** para imprimir somente o texto, sem exibir a estrutura bruta. Em ambos os casos, a resposta continua sendo gerada pelo LLM.

In [ ]:
# Chatbot: o modelo gera uma resposta para a mensagem.
relato_inicial = "O checkout retorna erro HTTP 500 após o deploy."
resposta_chatbot = modelo.invoke([
    HumanMessage(content=f"Resuma o sintoma em uma frase, sem afirmar a causa: {relato_inicial}")
])
# .text funciona com OpenAI (texto simples) e Gemini (blocos de conteúdo).
print("CHATBOT:", resposta_chatbot.text)

### Demonstração interativa — Chat com Gradio (slide 4)

Agora vamos usar **Gradio** como interface para conversar com o modelo escolhido na configuração: **OpenAI ou Gemini**. A interface visual é a mesma; apenas o provedor do LLM muda.

1. Envie **“O checkout retorna erro HTTP 500 após o deploy.”**
2. Pergunte **“Quais informações seriam necessárias para investigar?”** e observe o reaproveitamento do histórico da conversa.
3. Pergunte **“Você consultou as métricas do incidente 2048?”** — o chatbot **não dispõe de ferramenta de consulta** nesta etapa.

**Observe:** Gradio é somente a interface. `modelo.invoke()` faz uma chamada real ao LLM escolhido; verificar uma string vazia é uma regra Python determinística. A interface não transforma, por si só, um chatbot em agente.

> **Uso local:** a interface aparece no notebook ou informa um endereço `http://127.0.0.1:...`. A sessão depende do kernel. Evite `share=True`. A chave é guardada no ambiente da execução e **nunca deve ser enviada como mensagem do chat**.


In [ ]:
# Gradio fornece a interface visual; quem interpreta é o LLM configurado em `modelo`.
import gradio as gr
from langchain.messages import AIMessage

# No Gradio 6, o histórico pode trazer texto em blocos estruturados.
def extrair_texto(conteudo):
    if isinstance(conteudo, str):
        return conteudo
    if isinstance(conteudo, list):
        return "\n".join(
            bloco.get("text", "")
            for bloco in conteudo
            if isinstance(bloco, dict) and bloco.get("type") == "text"
        )
    return ""


def responder_chat(mensagem: str, historico: list) -> str:
    # Uma regra fixa que não precisa do LLM.
    if not mensagem.strip():
        return "Informe um relato ou uma pergunta para continuar."

    mensagens = [SystemMessage(content=(
        "Você é um chatbot didático de triagem de incidentes fictícios. "
        "Responda em português, de forma breve e clara. "
        "Não invente métricas, logs, causas ou ações realizadas. "
        "Você não tem acesso a ferramentas nem a sistemas de produção."
    ))]

    # Mantemos as últimas seis mensagens para dar continuidade à conversa.
    # Este histórico vale apenas para a sessão do chat, não é memória persistente.
    for item in historico[-6:]:
        texto = extrair_texto(item.get("content", ""))
        if not texto.strip():
            continue
        if item.get("role") == "user":
            mensagens.append(HumanMessage(content=texto))
        elif item.get("role") == "assistant":
            mensagens.append(AIMessage(content=texto))

    mensagens.append(HumanMessage(content=mensagem))
    resposta = modelo.invoke(mensagens)  # Uma chamada ao LLM a cada envio.
    # .text remove a estrutura de blocos e metadados sem depender do provedor.
    return str(resposta.text) or "O modelo não retornou uma resposta textual."


chat_gradio = gr.ChatInterface(
    fn=responder_chat,
    title="Detetive de Incidentes — Chatbot",
    description=(
        f"Modelo ativo: {ROTULO_PROVEDOR} ({MODELO_ID}). "
        "Demonstração da Aula 1: conversa com um LLM, sem acesso a ferramentas. "
        "Use relatos fictícios e não inclua dados sensíveis."
    ),
    examples=[
        "O checkout retorna erro HTTP 500 após o deploy.",
        "O catálogo está lento. O que devo investigar?",
        "Você consegue consultar os dados do incidente 2048?",
    ],
)

# A interface aparece no notebook e também fornece um endereço local.
# share=False evita criar um link público. Não execute esta célula repetidamente.
chat_gradio.launch(inline=True, share=False, prevent_thread_lock=True)


In [ ]:
# Workflow determinístico: as condições já foram definidas por nós.
def triagem_deterministica(texto: str) -> str:
    texto = texto.lower()
    if "500" in texto or "erro" in texto:
        return "Fila: investigar falha de aplicação"
    if "lento" in texto or "lentidão" in texto:
        return "Fila: investigar desempenho"
    return "Fila: solicitar mais informações"

print("WORKFLOW:", triagem_deterministica(relato_inicial))

## 2. Componentes essenciais no código (slide 7)

Mesmo quando a solução é pequena, podemos localizar os quatro componentes discutidos no slide:

- **LLM:** `modelo` interpreta o pedido e pode solicitar uma ação.
- **Instruções:** a mensagem de sistema define papel e limites.
- **Mensagens/estado:** uma estrutura simples guarda o histórico **desta execução**; não é memória persistente.
- **Ferramenta:** uma função Python permite consultar dados que não estão na mensagem do usuário.

Abaixo, a ferramenta consulta **dados simulados**. O modelo apenas solicita seu uso; **o Python é quem efetivamente executa a função**. Na Aula 2, o LangGraph vai coordenar esse ciclo.

In [ ]:
# FERRAMENTA: consulta fictícia, somente leitura.
@tool
def consultar_sinais(codigo: str) -> dict:
    """Consulta sinais fictícios de um incidente pelo código informado."""
    incidentes = {
        "2048": {"erros_pct": 18.4, "ultimo_deploy": "14:32"},
        "2049": {"erros_pct": 0.4, "ultimo_deploy": "12:10"},
    }
    return incidentes.get(codigo, {"erro": "Incidente não encontrado"})

# INSTRUÇÕES: o que o modelo pode e não pode fazer.
INSTRUCOES_AGENTE = (
    "Você auxilia na investigação de incidentes fictícios. "
    "Para consultar sinais de um incidente identificado por código, use a ferramenta disponível. "
    "Não invente logs, métricas ou causas. Não execute alterações. "
    "Responda brevemente em português."
)

# ESTADO/MENSAGENS: dados compartilhados durante este exemplo.
estado_demo = {
    "messages": [
        SystemMessage(content=INSTRUCOES_AGENTE),
        HumanMessage(content="Consulte os sinais do incidente 2048."),
    ]
}

# LLM + FERRAMENTAS: o modelo recebe o contrato da função.
ferramentas = [consultar_sinais]
modelo_com_ferramentas = modelo.bind_tools(ferramentas)
print("Componentes preparados:", ["LLM", "instruções", "mensagens", "ferramenta"])

In [ ]:
# O LLM DECIDE se solicita a ferramenta; este código demonstra a execução.
decisao = modelo_com_ferramentas.invoke(estado_demo["messages"])
print("Chamadas solicitadas pelo LLM:", decisao.tool_calls)

if decisao.tool_calls:
    estado_demo["messages"].append(decisao)
    for chamada in decisao.tool_calls:
        if chamada["name"] != "consultar_sinais":
            raise ValueError("Ferramenta não autorizada neste exemplo.")
        dados = consultar_sinais.invoke(chamada["args"])
        print("TOOL — dados consultados:", dados)
        estado_demo["messages"].append(
            ToolMessage(
                content=json.dumps(dados, ensure_ascii=False),
                tool_call_id=chamada["id"],
            )
        )
    resposta_agente = modelo_com_ferramentas.invoke(estado_demo["messages"])
else:
    resposta_agente = decisao

print("AGENTE — resposta:", resposta_agente.text)
# Este é um ciclo curto ilustrativo. O controle de chamadas repetidas fica para a Aula 2.

### O que observar antes do desafio?

Compare o comportamento dos três exemplos:

- O **chatbot** (inclusive pela interface Gradio) gerou texto sem consultar os dados simulados.
- O **workflow** seguiu os critérios do `if`, sem decidir autonomamente.
- Na demonstração de **agente**, o LLM pôde solicitar a função `consultar_sinais`, e o Python devolveu a observação.

**Atenção:** um relato que menciona um deploy não comprova que ele causou a falha.

### Desafio 1 — Um novo relato

Agora vamos mudar a solicitação. No código acima, vimos como enviar texto ao LLM e como passar o mesmo texto pela regra determinística.

**Sua tarefa:** use o relato **“O catálogo está lento após a atualização.”** e compare a resposta do chatbot com a classificação do workflow.

**Observe:** o que foi definido por código e o que foi redigido pelo modelo?

In [ ]:
# DESAFIO 1 — complete as chamadas usando o relato abaixo.
relato_desafio = "O catálogo está lento após a atualização."

# resposta_desafio = modelo.invoke([HumanMessage(content=...)])
# print("Chatbot:", ...)
# print("Workflow:", triagem_deterministica(...))

---
## 3. Instruções e contexto (slides 12–13)

Uma chamada ao LLM não basta para determinar seu comportamento. Vamos separar:

- **Instrução do sistema:** define função, objetivo e restrições.
- **Mensagem do usuário:** traz os dados da solicitação.

O **Detetive de Incidentes** deve identificar sintomas e indicar dados necessários para investigar. Ele **não pode afirmar que executou rollback** nem concluir uma causa sem evidências.

In [ ]:
INSTRUCOES = """
Você é um assistente de triagem de incidentes de software.
Use somente os fatos informados para resumir os sintomas observados.
Não invente causa raiz, logs, métricas ou resultados de consulta.
Não afirme que realizou rollback, deploy ou qualquer outra alteração.
Se faltarem evidências, indique objetivamente o que precisa ser investigado.
Responda em português e de forma breve.
""".strip()

relato_sem_evidencias = "O checkout falhou após o deploy. Descubra a causa agora."
resposta_orientada = modelo.invoke([
    SystemMessage(content=INSTRUCOES),
    HumanMessage(content=relato_sem_evidencias),
])
print(resposta_orientada.text)

### Desafio 2 — Respeitando os limites

O exemplo anterior mostrou que **instruções** e **dados da solicitação** não têm a mesma função.

**Sua tarefa:** envie o pedido abaixo com as mesmas instruções do sistema. Depois, identifique se a resposta afirma indevidamente que executou uma ação.

> “O serviço caiu. Faça o rollback imediatamente e confirme que resolveu.”

**Pergunta para discussão:** escrever “rollback realizado” seria evidência de uma operação real?

In [ ]:
# DESAFIO 2 — construa as mensagens e execute a chamada.
solicitacao_sensivel = "O serviço caiu. Faça o rollback imediatamente e confirme que resolveu."

# mensagens_desafio = [SystemMessage(content=INSTRUCOES), ...]
# resposta_desafio_2 = modelo.invoke(mensagens_desafio)
# print(...)

---
## 4. Contrato de saída: Pydantic (slide 14)

Até aqui o LLM respondeu em **texto livre**. Para que outra parte do sistema utilize o resultado, precisamos de campos previsíveis.

Nosso contrato terá quatro campos:

| Campo | Tipo esperado | Para quê? |
|---|---|---|
| `tipo` | `erro`, `lentidao`, `indefinido` | Identificar o sintoma |
| `gravidade` | `alta`, `normal`, `indefinida` | Registrar se o impacto foi informado |
| `resumo` | Texto com 5 caracteres ou mais | Sintetizar o relato |
| `precisa_consulta` | `True` ou `False` | Sinalizar necessidade de evidências |

**Contrato de saída não é um prompt longo:** é a definição de quais campos e valores a aplicação aceita.

In [ ]:
class AnaliseIncidente(BaseModel):
    tipo: Literal["erro", "lentidao", "indefinido"] = Field(
        description="Sintoma presente no relato; indefinido se não houver dados suficientes."
    )
    gravidade: Literal["alta", "normal", "indefinida"] = Field(
        description=(
            "Classifique como alta quando houver evidência explícita "
            "de impacto significativo; normal quando houver indicação "
            "explícita de baixo impacto; indefinida quando não existirem "
            "informações suficientes para determinar a gravidade. "
            "Não deduza gravidade apenas pelo tipo de erro ou latência."
        )
    )
    resumo: str = Field(
        min_length=5,
        description="Resumo breve do sintoma, sem presumir causa raiz."
    )
    precisa_consulta: bool = Field(
        description=(
            "True quando são necessárias evidências adicionais "
            "para investigar a causa ou confirmar o diagnóstico. "
            "Classificar o sintoma não elimina essa necessidade."
        )
    )

# Neste momento, estamos validando um objeto Python — SEM chamar o LLM.
registro_valido = {
    "tipo": "erro",
    "gravidade": "indefinida",
    "resumo": "Checkout retorna HTTP 500.",
    "precisa_consulta": True,
}
registro = AnaliseIncidente.model_validate(registro_valido)
print("Objeto validado:", registro.model_dump())
print("Campo acessado:", registro.tipo)

In [ ]:
# E se o objeto não respeitar o contrato? O Pydantic informa o erro.
registro_invalido = {
    "tipo": "desconhecido",  # não pertence aos valores permitidos
    "gravidade": "alta",
    "resumo": "ops",        # menos de 5 caracteres
    "precisa_consulta": "talvez",  # não é booleano válido
}

try:
    AnaliseIncidente.model_validate(registro_invalido)
except ValidationError as erro:
    for detalhe in erro.errors():
        print("Campo:", detalhe["loc"], "| Regra:", detalhe["type"])

### Aplicando o contrato ao LLM (slide 15)

Agora solicitaremos ao modelo uma resposta que siga `AnaliseIncidente`. Assim, recebemos um **objeto validável**, não apenas um parágrafo.

**Importante:** cumprir o formato **não garante que o diagnóstico esteja correto**. Precisamos continuar comparando a resposta com as evidências. O suporte a saída estruturada depende do provedor e do modelo configurados. Os exemplos usam o mesmo método `with_structured_output()` para OpenAI e Gemini.

In [ ]:
classificador = modelo.with_structured_output(AnaliseIncidente)

analise = classificador.invoke([
    SystemMessage(content=INSTRUCOES),
    HumanMessage(content=(
        "Incidente 2048: o checkout apresenta erro HTTP 500 após o deploy."
    )),
])

print(analise.model_dump_json(indent=2))
print("Tipo:", analise.tipo)
print("Precisa consultar dados externos?", analise.precisa_consulta)

### Desafio 3 — Classificando um incidente de lentidão

No exemplo anterior, o relato mencionava **HTTP 500**. Agora use o mesmo contrato para uma ocorrência de **lentidão**.

**Sua tarefa:** analise **“O checkout está levando 12 segundos para concluir o pedido.”** e mostre os campos `tipo` e `precisa_consulta`.

**Observe:** o modelo sinalizou corretamente a lentidão? Há evidência suficiente para afirmar a causa?

In [ ]:
# DESAFIO 3 — utilize o classificador já criado.
relato_lentidao = "O checkout está levando 12 segundos para concluir o pedido."

# analise_lentidao = classificador.invoke([SystemMessage(...), HumanMessage(...)])
# print(analise_lentidao.model_dump_json(indent=2))
# print("Tipo:", ...)
# print("Precisa consultar?", ...)

---
## 5. Uma função reutilizável e testes iniciais (slides 16–19)

Para não repetir toda a chamada, vamos definir uma função. **Esta será a principal entrega da Aula 1.**

Em seguida, testaremos um relato claro e outro ambíguo. O objetivo não é apenas verificar se a execução terminou sem erro: queremos conferir se o conteúdo respeita o que sabemos.

In [ ]:
def analisar_incidente(texto: str) -> AnaliseIncidente:
    if not texto.strip():
        raise ValueError("Informe um relato para análise.")
    return classificador.invoke([
        SystemMessage(content=INSTRUCOES),
        HumanMessage(content=texto),
    ])

casos_demonstracao = {
    "Erro claro": "O checkout retorna HTTP 500 para diversos usuários.",
    "Relato ambíguo": "O sistema está estranho.",
}

for nome, relato in casos_demonstracao.items():
    resultado = analisar_incidente(relato)
    print(f"\n--- {nome} ---")
    print(resultado.model_dump_json(indent=2))

### Desafio 4 — Uma pequena bateria de testes

Acabamos de analisar um erro claro e uma solicitação ambígua. Agora você vai executar **três novos cenários**.

1. **Lentidão:** “O catálogo demora mais de 10 segundos para carregar.”
2. **Falta de contexto:** “Está tudo ruim.”
3. **Erro genérico:** “O serviço de autenticação retorna erro durante o login.”

**Sua tarefa:** utilize `analisar_incidente` em um laço e apresente o `tipo`, o `resumo` e a indicação de consulta. Compare os resultados com os relatos. Não há uma única classificação de gravidade obrigatória quando o impacto não foi informado.

In [ ]:
# DESAFIO 4 — use o laço apresentado na célula anterior.
casos_para_testar = {
    "Lentidão": "O catálogo demora mais de 10 segundos para carregar.",
    "Falta de contexto": "Está tudo ruim.",
    "Erro genérico": "O serviço de autenticação retorna erro durante o login.",
}

# for nome, relato in casos_para_testar.items():
#     resultado = analisar_incidente(...)
#     print(...)
#     ...

---
## Checkpoint — o que conseguimos construir?

- [ ] Sei explicar por que **chatbot, workflow determinístico e agente** não são a mesma coisa.
- [ ] Consigo localizar **LLM, instruções, mensagens/estado e ferramenta** nos exemplos.
- [ ] Fiz uma chamada ao modelo e testei o chat com Gradio, sem expor minha chave.
- [ ] Consigo explicar o que o **Pydantic valida** e o que ele **não garante**.
- [ ] Executei os desafios e testei ao menos um caso ambíguo.
- [ ] Tenho uma função `analisar_incidente(texto)` utilizável na próxima aula.

**Na Aula 2:** vamos organizar a execução com **LangGraph**, estado, nós, transições e seleção de ferramentas. O experimento de consulta simulada de hoje ajudará a reconhecer essas partes no grafo.

### Se algo não funcionar

- **Seleção de provedor:** na célula de configuração, digite `1` para OpenAI ou `2` para Gemini. Se pressionar Enter, o padrão será Gemini. Para escolher novamente, reexecute essa célula.
- **Erro de autenticação:** verifique `OPENAI_API_KEY` (OpenAI) ou `GOOGLE_API_KEY` / `GEMINI_API_KEY` (Gemini). A chave deve corresponder ao provedor escolhido. Se houver as duas variáveis Google, **`GOOGLE_API_KEY` tem prioridade**. Nunca imprima nem compartilhe o valor da chave.
- **Modelo indisponível:** para OpenAI, altere `OPENAI_MODEL`; para Gemini, altere `GEMINI_MODEL`. O padrão é `gpt-4.1-mini` ou `gemini-3.5-flash-lite`, respectivamente. Confira no console do provedor quais modelos sua chave permite usar.
- **Limite de cota / HTTP 429:** o provedor pode restringir requisições por minuto, por dia ou conforme o plano. Diminua a frequência e consulte o painel de uso; **uma API key válida não garante cota disponível**.
- **Biblioteca ausente:** execute novamente a instalação `%pip install ...` e, se necessário, reinicie o kernel antes de recomeçar.
- **Gradio não abre:** confira se a instalação terminou, execute a célula do chat apenas uma vez e use o endereço local informado. Em ambientes remotos, a porta local pode ser inacessível. Para fechar uma instância anterior, execute `chat_gradio.close()`.
- **Mudança de provedor depois de executar o notebook:** execute novamente as células que constroem `modelo_com_ferramentas` e `classificador`; elas não mudam sozinhas quando `modelo` é substituído.
- **Saída estruturada não suportada:** confirme que o modelo escolhido suporta esse recurso e que sua versão da integração está atualizada; a compatibilidade depende do modelo/provedor.
- **Ferramenta não chamada:** verifique `decisao.tool_calls`. A seleção de ferramentas pelo LLM é probabilística, não uma garantia de execução.
- **Campos inesperados:** compare o relato, as instruções e o contrato; **formato válido não comprova diagnóstico correto**.

**Referências:** [LangChain Python](https://docs.langchain.com/oss/python/langchain/overview) · [Integração Gemini (LangChain)](https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai) · [Google AI Studio](https://aistudio.google.com/api-keys) · [Pydantic](https://docs.pydantic.dev/latest/) · [Gradio](https://www.gradio.app/docs/gradio/chatinterface).

- **Gemini exibindo `[{'type': 'text', ...}]`:** isso ocorre ao imprimir `mensagem.content`, que pode conter blocos e metadados. Use `mensagem.text` para exibir somente o texto, tanto na OpenAI quanto no Gemini. Não copie assinaturas ou metadados para os slides.
- **Causa não comprovada:** um relato de erro após o deploy não prova que o deploy causou a falha; o texto gerado deve distinguir correlação temporal de diagnóstico confirmado.
